# Huấn luyện QA-Graph cho VG và PSG trên Colab GPU

Hai bundle đã có `huy_extra_cn` trên MyDrive. Notebook chạy lại config đã báo cáo của Huy: VG `t1_qa_cn_no_sg_sage` (SAGE), PSG `psg_qa_cn` (GAT, gồm scene graph + ConceptNet). Ba seed mỗi bộ được lưu thành sáu checkpoint riêng, cùng config, α chọn trên validation, ranks test và SHA256 dữ liệu/model. Không huấn luyện lại CLIP. Mở `Runtime → Change runtime type → T4 GPU`, rồi chạy tuần tự. Đồng bộ mã mới của `duong/xai/model.py` lên repo ở Drive trước khi chạy.

Đầu ra: `MyDrive/graphml-project3/duong/runs/qa_vg_psg_xai/<experiment>/<run_id>/` và một manifest JSON ở thư mục cha. Notebook không xóa kết quả cũ; XAI notebook sẽ dọn đúng hai thư mục artifact cũ sau khi kết quả mới hoàn tất.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, importlib.util, json, os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')
REPO = DRIVE / 'graphml-project3'
RUNS = REPO / 'duong/runs/qa_vg_psg_xai'
SEEDS = (0, 1, 2)
RUN_LABEL = datetime.now(timezone.utc).strftime('dual-%Y%m%d-%H%M%S')
assert (REPO / 'huy/src/mmxai/graph/qa.py').is_file()
assert (REPO / 'duong/xai/model.py').is_file(), 'Đồng bộ repo mới lên Drive trước khi chạy.'
if importlib.util.find_spec('torch_geometric') is None or importlib.util.find_spec('yaml') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO / 'huy')], check=True)
import torch
assert torch.cuda.is_available(), 'Chọn GPU trong Colab Runtime.'
sys.path.insert(0, str(REPO / 'duong'))
from xai.model import MODEL_SPECS
RUNS.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), '| run:', RUN_LABEL)


In [ ]:
BUNDLES = {name: DRIVE / spec.bundle_dir for name, spec in MODEL_SPECS.items()}
FILES = ('data/images.jsonl', 'data/captions.jsonl', 'data/object_vocab_train.json',
         'features/meta.json', 'features/clip_ViT-B-32_laion2b.npz', 'split/split.json',
         'mmxai_eval.py', 'huy_extra_cn/labels.json', 'huy_extra_cn/label_emb.npy',
         'huy_extra_cn/conceptnet.tsv')
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

SOURCE_FILES = ('huy/src/mmxai/graph/qa.py', 'huy/src/mmxai/graph/build.py',
                'huy/src/mmxai/data/task1.py', 'huy/src/mmxai/experiments/t1_qa.py',
                'duong/xai/model.py', 'duong/xai/perturbation.py',
                'duong/xai/explain.py', 'duong/xai/evaluate_xai.py',
                'duong/xai/scorer_adapter.py', 'duong/xai/graph.py', 'duong/xai/io.py')
source_fingerprints = {rel: sha256(REPO / rel) for rel in SOURCE_FILES}
fingerprints = {}
for name, spec in MODEL_SPECS.items():
    bundle = BUNDLES[name]
    missing = [rel for rel in FILES if not (bundle / rel).is_file()]
    assert not missing, f'{name}: thiếu {missing} trong {bundle}'
    meta = json.loads((bundle / 'features/meta.json').read_text())
    labels = json.loads((bundle / 'huy_extra_cn/labels.json').read_text())
    edges = sum(1 for _ in (bundle / 'huy_extra_cn/conceptnet.tsv').open())
    assert meta['model'] == 'ViT-B-32' and meta['pretrained'] == 'laion2b_s34b_b79k'
    assert len(labels) == spec.expected_labels and 2 * edges == spec.expected_concept_edges
    fingerprints[name] = {rel: sha256(bundle / rel) for rel in FILES}
    print(name, '|', bundle, '| labels:', len(labels), '| directed CN edges:', 2 * edges)


In [ ]:
run_records = []
for name, spec in MODEL_SPECS.items():
    for seed in SEEDS:
        run_id = f'{RUN_LABEL}_{name}_seed{seed}'
        run_dir = RUNS / spec.experiment / run_id
        assert not run_dir.exists(), f'Run đã tồn tại: {run_dir}'
        cmd = [sys.executable, '-m', 'mmxai.run', '--exp', spec.experiment,
               '--config-dir', str(REPO / 'huy/configs'), '--seed', str(seed),
               '--set', f'data={BUNDLES[name]}',
               '--set', f'extra={BUNDLES[name] / "huy_extra_cn"}',
               '--set', 'save_model=true', '--runs-dir', str(RUNS),
               '--run-id', run_id, '--no-publish']
        env = os.environ.copy()
        env['LAB_USER'] = 'duong'
        env['PYTHONHASHSEED'] = str(seed)
        env['PYTHONPATH'] = str(REPO / 'huy/src') + os.pathsep + env.get('PYTHONPATH', '')
        print('Running', name, 'seed', seed, flush=True)
        subprocess.run(cmd, cwd=REPO / 'huy', env=env, check=True)
        checkpoint = run_dir / 'model.pt'
        assert checkpoint.is_file() and checkpoint.stat().st_size > 0
        metrics = json.loads((run_dir / 'metrics.json').read_text())
        assert metrics['run']['error'] is None, metrics['run']['error']
        cfg = json.loads((run_dir / 'config.json').read_text())
        stats = metrics['run']['extra']['graph']
        assert cfg['graph']['variant'] == spec.graph_variant and cfg['gnn']['kind'] == spec.gnn_kind
        assert stats['n_label'] == spec.expected_labels and stats['E_concept'] == spec.expected_concept_edges
        info = json.loads((run_dir / 'train_info.json').read_text())[0]
        record = {'dataset': name, 'experiment': spec.experiment, 'seed': seed,
                  'run_dir': str(run_dir), 'checkpoint_sha256': sha256(checkpoint),
                  'alpha': info['alpha'], 'test_full_MRR': metrics['metrics']['test_full']['MRR']['mean']}
        run_records.append(record)
        print('Saved', checkpoint, '| MRR', record['test_full_MRR'], '| alpha', record['alpha'])

manifest = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'run_label': RUN_LABEL,
            'bundle_sha256': fingerprints, 'source_sha256': source_fingerprints, 'runs': run_records}
manifest_path = RUNS / f'{RUN_LABEL}_manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
print('Manifest:', manifest_path)


## Sau khi huấn luyện

Kiểm tra manifest có đủ sáu lượt và `model.pt` của từng lượt. Sau đó chạy `xai_vg_psg_colab.ipynb`, notebook này sẽ nạp đúng bundle, config, checkpoint và α theo manifest. Dữ liệu test chỉ dùng để báo cáo và chọn case study cố định; checkpoint không được chọn theo MRR test.
